<a href="https://colab.research.google.com/github/Gedhang10/Saham_Streamlit/blob/main/saham3persen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install library terlebih dahulu jika belum
!pip install yfinance pandas_ta pandas numpy scikit-learn xgboost matplotlib streamlit

import yfinance as yf
import pandas_ta as ta
import pandas as pd
import numpy as np
import streamlit as st
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Mengambil Data Saham
def get_stock_data(symbol, period='1y', interval='1d'):
    try:
        data = yf.download(symbol, period=period, interval=interval)
        return data
    except Exception as e:
        st.error(f"Error mendapatkan data untuk {symbol}: {e}")
        return pd.DataFrame()

# 2. Menambahkan Indikator Teknikal
def add_indicators(data):
    if data.empty:
        return data

    data['SMA_50'] = ta.sma(data['Close'], length=50)
    data['EMA_12'] = ta.ema(data['Close'], length=12)
    data['RSI'] = ta.rsi(data['Close'], length=14)
    macd = ta.macd(data['Close'])
    data['MACD'] = macd['MACD_12_26_9'] if macd is not None else None
    bb = ta.bbands(data['Close'], length=20, std=2)
    data['BB_upper'] = bb['BBU_20_2.0'] if bb is not None else None
    data['BB_lower'] = bb['BBL_20_2.0'] if bb is not None else None
    data['ATR'] = ta.atr(data['High'], data['Low'], data['Close'], length=14)
    return data.fillna(0)

# 3. Melatih Model
def train_models(data):
    data = add_indicators(data)
    X = data[['SMA_50', 'EMA_12', 'RSI', 'MACD', 'BB_upper', 'BB_lower', 'ATR']].dropna()
    y = data['Close']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf_model = RandomForestRegressor()
    rf_model.fit(X_train_scaled, y_train)
    xgb_model = XGBRegressor(objective='reg:squarederror')
    xgb_model.fit(X_train_scaled, y_train)

    rf_pred = rf_model.predict(X_test_scaled)
    xgb_pred = xgb_model.predict(X_test_scaled)

    mse_rf = mean_squared_error(y_test, rf_pred)
    r2_rf = r2_score(y_test, rf_pred)
    mse_xgb = mean_squared_error(y_test, xgb_pred)
    r2_xgb = r2_score(y_test, xgb_pred)

    return rf_model, xgb_model, mse_rf, r2_rf, mse_xgb, r2_xgb

# 4. Menampilkan Prediksi
def display_predictions(symbol, data, rf_model, xgb_model):
    features = ['SMA_50', 'EMA_12', 'RSI', 'MACD', 'BB_upper', 'BB_lower', 'ATR']
    latest_data = data[features].iloc[-1].values.reshape(1, -1)
    rf_pred = rf_model.predict(latest_data)
    xgb_pred = xgb_model.predict(latest_data)

    st.write(f"Prediksi harga saham {symbol}:")
    st.write(f"- Random Forest: {rf_pred[0]:.2f}")
    st.write(f"- XGBoost: {xgb_pred[0]:.2f}")
    return rf_pred, xgb_pred

# 5. Visualisasi Prediksi
def plot_results(data, rf_pred, xgb_pred):
    plt.figure(figsize=(10, 5))
    plt.plot(data['Close'], label='Harga Penutupan')
    plt.axvline(x=data.index[-1], color='red', linestyle='--', label='Prediksi')
    plt.scatter(data.index[-1], rf_pred, color='green', label='Prediksi RF')
    plt.scatter(data.index[-1], xgb_pred, color='orange', label='Prediksi XGB')
    plt.legend()
    st.pyplot(plt)

# 6. Streamlit UI
def main():
    st.title("Prediksi Saham dengan RandomForest dan XGBoost")
    symbol = st.text_input("Masukkan simbol saham:", "AAPL")
    period = st.selectbox("Pilih periode:", ["1mo", "3mo", "6mo", "1y", "5y"])

    if st.button("Prediksi"):
        data = get_stock_data(symbol, period)
        if not data.empty:
            rf_model, xgb_model, mse_rf, r2_rf, mse_xgb, r2_xgb = train_models(data)
            st.write(f"Evaluasi Model:")
            st.write(f"- Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}")
            st.write(f"- XGBoost MSE: {mse_xgb:.2f}, R2: {r2_xgb:.2f}")

            rf_pred, xgb_pred = display_predictions(symbol, data, rf_model, xgb_model)
            plot_results(data, rf_pred, xgb_pred)
        else:
            st.error("Gagal mendapatkan data.")

if __name__ == "__main__":
    main()


2024-12-18 09:03:11.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.367 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.368 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.370 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.372 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.376 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-18 09:03:11.377 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar